In [1]:
%matplotlib widget
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
import sys
import datetime
from tqdm import tqdm

%load_ext autoreload
%autoreload 2

sys.path.append('/home/wolfgang/repos/sleep_general')
from mgh_sleeplab import load_mgh_signal, annotations_preprocess, vectorize_respiratory_events, vectorize_sleep_stages

In [2]:
table_grass = pd.read_csv('/media/mad3/Datasets_ConvertedData/sleeplab/grass_studies_list.csv')
table_natus = pd.read_csv('/media/mad3/Datasets_ConvertedData/sleeplab/natus_studies_list.csv')
assert np.all(table_grass.columns == table_natus.columns)
sleeplab_table = pd.concat([table_grass, table_natus], axis=0)
sleeplab_table = sleeplab_table[pd.notna(sleeplab_table.MRN)]
sleeplab_table.reset_index(inplace=True, drop=True)
print(sleeplab_table.shape)

(25231, 10)


In [3]:
print('TMP')
part = 2
sleeplab_table = sleeplab_table.iloc[8700*part : 8700*(part+1)].copy()
print(sleeplab_table.shape)

TMP
(7831, 10)


In [4]:
sleeplab_table['path_signal'] = np.nan
sleeplab_table['path_annotations'] = np.nan

for jloc, row in sleeplab_table.iterrows():
    
    try:
        path = row.Path
        path = path.replace('M:', '/media/mad3')
        path = path.replace('\\', '/')

        path_signalfile = np.nan
        signalfile = [x for x in os.listdir(path) if 'Signal' in x]
        if len(signalfile) > 0:
            signalfile = signalfile[0]
            path_signalfile = os.path.join(path, signalfile)

        path_annotations = np.nan
        if 'annotations.csv' in os.listdir(path):
            path_annotations = os.path.join(path, 'annotations.csv')

        sleeplab_table.loc[jloc, 'path_signal'] = path_signalfile
        sleeplab_table.loc[jloc, 'path_annotations'] = path_annotations

    except Exception as e:
        print(e)
        continue
        
sleeplab_table.to_csv(f'sleeplab_table_{part}.csv')

In [5]:
print('TMP')
sleeplab_table = sleeplab_table.iloc[:8000].copy()
print(sleeplab_table.shape)

TMP
(7831, 12)


In [6]:
from sleep_analysis_functions import compute_spo2_clean, compute_hypoxia_burden, desaturation_detection, hypoxaemic_burden_minutes

def compute_hypoxia_statistics(data, apnea, sleep_stage, fs):
    
    data['apnea'] = apnea

    dt_start = pd.Timestamp('2000-01-01 00:00:00')
    dt_end = dt_start + datetime.timedelta(seconds=(data.shape[0]-1) / fs)
    pseudo_dt_index = pd.date_range(start=dt_start, end=dt_end, periods=data.shape[0])
    data.index = pseudo_dt_index

    data = compute_spo2_clean(data, fs=fs)
    data['spo2'] = data['spo2_clean']

    data['apnea_binary'] = np.isin(data['apnea'],[1,2,3,4]).astype(int)
    data['apnea_end'] = np.isin(data['apnea_binary'].diff(), [-1])
    
    sleep_stage = sleep_stage[np.logical_not(pd.isna(sleep_stage))]
    hours_sleep = sum(sleep_stage<5)/fs/3600
    
    data, hypoxia_burden = compute_hypoxia_burden(data, fs, hours_sleep=hours_sleep, apnea_name = 'apnea')
    
    T90burden, T90desaturation, T90nonspecific = hypoxaemic_burden_minutes(data['spo2'].values, fs)
    
    AHI = np.round(sum(data['apnea_end'])/hours_sleep,1)

    return hypoxia_burden, AHI, hours_sleep, T90burden, T90desaturation, T90nonspecific


In [7]:
sleeplab_table.shape

(7831, 12)

In [11]:
sleeplab_table['Total_Sleep_Time'] = np.nan
sleeplab_table['AHI'] = np.nan
sleeplab_table['HypoxiaBurden'] = np.nan
sleeplab_table['T90_minutes'] = np.nan
sleeplab_table['T90_minutes_desat'] = np.nan
sleeplab_table['T90_minutes_unspecific'] = np.nan

for jloc, row in sleeplab_table.iterrows():
    
    try:
        if pd.isna(row.path_signal) | pd.isna(row.path_annotations):
            continue
        # read in raw data:
        signal, params = load_mgh_signal(row.path_signal)
        annotations = pd.read_csv(row.path_annotations)

        fs = int(params['Fs'])
        signal_len = signal.shape[0]

        # get respiratory event and sleep stage array:
        annotations = annotations_preprocess(annotations, fs)
        resp = vectorize_respiratory_events(annotations, signal_len)
        sleep_stage = vectorize_sleep_stages(annotations, signal_len)

        hypoxia_burden, ahi, total_sleep_time, T90burden, T90desaturation, T90nonspecific = compute_hypoxia_statistics(signal, resp, sleep_stage, fs)

        sleeplab_table.loc[jloc, 'AHI'] = ahi
        sleeplab_table.loc[jloc, 'HypoxiaBurden'] = hypoxia_burden
        sleeplab_table.loc[jloc, 'Total_Sleep_Time'] = total_sleep_time
        sleeplab_table.loc[jloc, 'T90_minutes'] = T90burden
        sleeplab_table.loc[jloc, 'T90_minutes_desat'] = T90desaturation
        sleeplab_table.loc[jloc, 'T90_minutes_unspecific'] = T90nonspecific

        sleeplab_table.to_csv(f'sleeplab_table_hypoxia_{part}.csv')
        
    except Exception as e:
        print(f'{jloc}, {row.Path}')
        print(e)

sleeplab_table.to_csv(f'sleeplab_table_hypoxia_{part}.csv')

/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,


17424, M:\Datasets_ConvertedData\sleeplab\grass_data\Matour_Stephanie_010714_0838.000
index 5499200 is out of bounds for axis 0 with size 5499200


/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a

17447, M:\Datasets_ConvertedData\sleeplab\grass_data\Piretti_Ronald_022613_2244.000
float division by zero


/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a

...

23490, M:\Datasets_ConvertedData\sleeplab\natus_data\Horan~ Matthew_422c7900-c6cd-443e-a063-ede36a50cf6c
'DataFrame' object has no attribute 'duration'
23498, M:\Datasets_ConvertedData\sleeplab\natus_data\McGann~ Kevin_89e60075-2aff-4b25-8e8c-f9a34685c47a
'DataFrame' object has no attribute 'duration'
23502, M:\Datasets_ConvertedData\sleeplab\natus_data\GonzalezUribe~_6ac59440-f497-4f62-beee-e456f70c5f17
'DataFrame' object has no attribute 'duration'
23504, M:\Datasets_ConvertedData\sleeplab\natus_data\Poole~ Edith_fadb7033-9a62-40a4-91a1-ba19615ec1d1
iteration over a 0-d array
23506, M:\Datasets_ConvertedData\sleeplab\natus_data\Good~ Jonathan_f494f74f-1234-4409-944a-d85da72d67cc
'DataFrame' object has no attribute 'duration'
23509, M:\Datasets_ConvertedData\sleeplab\natus_data\Pappaglanopoul_cdd22901-e90d-4428-b346-91571799e42c
'DataFrame' object has no attribute 'duration'
23513, M:\Datasets_ConvertedData\sleeplab\natus_data\Devin~ Audrey_99a39d1e-7a88-4768-aec4-23261a08da95
iterati

<ipython-input-6-219ea860a567>:25: RuntimeWarning: invalid value encountered in double_scalars
  AHI = np.round(sum(data['apnea_end'])/hours_sleep,1)


23709, M:\Datasets_ConvertedData\sleeplab\natus_data\Slinger~ Seamu_65ee8662-9e85-4713-ac15-805ce43f1c95
'DataFrame' object has no attribute 'duration'
23717, M:\Datasets_ConvertedData\sleeplab\natus_data\Lamm~ SusanS_87f1f527-0f24-48c9-abd7-ec75750727f6
Cannot convert non-finite values (NA or inf) to integer
23725, M:\Datasets_ConvertedData\sleeplab\natus_data\Coriale~ Danie_f2417d8a-51ea-4c95-ae06-7a867051a3c9
index 12738632 is out of bounds for axis 0 with size 12738632
23730, M:\Datasets_ConvertedData\sleeplab\natus_data\Wilson~ Kimber_b1f995a2-dcac-4c90-aea3-3dd41281cb96
'DataFrame' object has no attribute 'duration'
23734, M:\Datasets_ConvertedData\sleeplab\natus_data\Thorpe~ AnnaMa_e9c37726-5ddb-4111-82c1-5cd0e23b1038
'DataFrame' object has no attribute 'duration'
23736, M:\Datasets_ConvertedData\sleeplab\natus_data\Maguire~ Patri_28ecadcf-cfb2-48eb-8ac2-19629ba67da0
'DataFrame' object has no attribute 'duration'
23739, M:\Datasets_ConvertedData\sleeplab\natus_data\TellezAguilar

<ipython-input-6-219ea860a567>:25: RuntimeWarning: invalid value encountered in double_scalars
  AHI = np.round(sum(data['apnea_end'])/hours_sleep,1)


24496, M:\Datasets_ConvertedData\sleeplab\natus_data\Smith~ Tina_0b1a65da-9fe7-4551-aa51-82b2bb678a8a
'DataFrame' object has no attribute 'duration'
24504, M:\Datasets_ConvertedData\sleeplab\natus_data\Wang~ Xuntuo_5eb88ff6-aa47-44de-851a-6d508f2d0051
'DataFrame' object has no attribute 'duration'
24506, M:\Datasets_ConvertedData\sleeplab\natus_data\Gibbons~ Joshu_a93dcef5-7395-4dbd-a458-7addda9ab2c7
index 11558401 is out of bounds for axis 0 with size 11558401
24510, M:\Datasets_ConvertedData\sleeplab\natus_data\Watkins~ Virgi_f7cb8823-9f30-4db3-9625-a25ca37e3957
'DataFrame' object has no attribute 'duration'
24512, M:\Datasets_ConvertedData\sleeplab\natus_data\Skinner~ Paul_c2c333d8-a755-4c29-8ed4-a4820732dd1a
'DataFrame' object has no attribute 'duration'
24516, M:\Datasets_ConvertedData\sleeplab\natus_data\Ho~ Hok_f30e3270-384e-425a-ac51-dfbe96d9e8f3
'DataFrame' object has no attribute 'duration'
24521, M:\Datasets_ConvertedData\sleeplab\natus_data\Quinn~ James_96080fae-9dbf-49dd-8

In [9]:
sleeplab_table

,FolderName,PatientID,MRN,LastName,FirstName,Sex,DateOfBirth,DateOfVisit,TypeOfTest,Path,path_signal,path_annotations,Total_Sleep_Time,AHI,HypoxiaBurden,T90_minutes,T90_minutes_desat,T90_minutes_unspecific
17400,Gomez_George_081710_2148.000,Z10369711,327-97-03,Gomez,George L,Male,1939-07-30,2010-08-17,PSG Diagnostic,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,7.575000,8.6,21.0,5.931333,5.065167,0.866167
17401,Corlito_William_111011_2246.000,Z7706850,083-27-16,Corlito,William A,Male,1952-06-01,2011-11-10,PSG Split night,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,6.325000,19.8,54.4,32.107833,27.676833,4.431000
17402,Bibeau_Ralph_043012_2349.000,Z10821997,254-39-33,Bibeau,Ralph H,Male,1937-04-19,2012-04-30,PSG Diagnostic,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,3.791667,6.6,3.1,0.216583,0.000000,0.216583
17403,Davis_Pamela_071609_2305.000,Z7200409,474-07-33,Davis,Pamela,Female,1955-06-02,2009-07-16,PSG Diagnostic,M:\Datasets_ConvertedData\sleeplab\grass_data\...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17404,Mercer_Daniel_070609_2240.000,Z10505605,331-76-02,Mercer,Daniel,Male,1967-07-11,2009-07-06,PSG Diagnostic,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25226,Tolls~ Volker_61b33222-aa32-4e56-af68-9de53f79...,Z6528174,354-68-42,Tolls,Volker,Male,1961-10-26,2018-11-06,Diagnostic,M:\Datasets_ConvertedData\sleeplab\natus_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/na...,/media/mad3/Datasets_ConvertedData/sleeplab/na...,NaN,NaN,NaN,NaN,NaN,NaN
25227,Riley~ Barita_822b1819-24cf-4ab4-990b-3a9529ed...,Z7772409,195-79-91,Riley,Barita A,Female,1961-12-04,2019-06-29,Split Night,M:\Datasets_ConvertedData\sleeplab\natus_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/na...,/media/mad3/Datasets_ConvertedData/sleeplab/na...,NaN,NaN,NaN,NaN,NaN,NaN
25228,Joseph~ Sonia_596b668a-5bc2-4c2d-b64c-52ee6db2...,NaN,501-73-39,Joseph,Sonia,Female,1963-12-16,2020-02-08,NaN,M:\Datasets_ConvertedData\sleeplab\natus_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/na...,/media/mad3/Datasets_ConvertedData/sleeplab/na...,NaN,NaN,NaN,NaN,NaN,NaN
25229,Theriault~ The_b6618d3b-1d75-43cb-a7ec-c858ad2...,Z9860819,313-31-78,Theriault,Therese C,Female,1947-10-17,2019-12-21,Titration CPAP,M:\Datasets_ConvertedData\sleeplab\natus_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/na...,/media/mad3/Datasets_ConvertedData/sleeplab/na...,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
# fig, ax = plt.subplots(4, 1, figsize=(10, 8), sharex=True)

# ax[0].plot(resp)
# ax[0].plot(sleep_stage)
# ax[1].plot(signal['spo2'])
# ax[1].set_ylim([80,100])
# # ax[2].plot(signal['ptaf'])
# ax[3].plot(signal['chest'])
# ax[3].plot(signal['abd'])